In [7]:
# Week 7 — HealthConnect KPI Validation & Testing
# Track: Data Analytics
# Intern: Wisdom Chibuike Ukah

import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = "HealthConnect_Appointment_Data.csv"

df = pd.read_csv('HealthConnect_Appointment_Data.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (5000, 21)
Columns: ['appointment_id', 'patient_id', 'gender', 'age', 'age_group', 'appointment_type', 'booking_date', 'appointment_date', 'appointment_day', 'appointment_time', 'booking_lead_days', 'Booking lead days category', 'previous_appointments', 'previous_no_shows', 'reminder_sent', 'reminder_channel', 'distance_to_clinic_km', 'Distance to clinic group', 'waiting_time_minutes', 'Waiting time group', 'appointment_outcome']


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,...,Booking lead days category,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,Distance to clinic group,waiting_time_minutes,Waiting time group,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,...,1-2 Weeks,2,0,Yes,WhatsApp,19.3,Moderate Distance,29.0,Moderate,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,...,Short Term,6,0,Yes,SMS,14.3,Moderate Distance,42.0,Long,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,...,1-2 Months,5,1,Yes,SMS,11.4,Moderate Distance,11.0,Short,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,...,1-2 Months,3,1,Yes,SMS,7.4,Close Distance,35.0,Long,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,...,1-2 Months,3,1,Yes,Email,5.6,Close Distance,27.0,Moderate,No-Show


In [8]:
df['reminder_channel'].value_counts(dropna=False)

reminder_channel
SMS         2000
NaN         1366
WhatsApp    1101
Email        533
Name: count, dtype: int64

In [9]:
df['appointment_outcome'].value_counts()

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

In [10]:
df.shape

(5000, 21)

In [11]:
df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,...,Booking lead days category,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,Distance to clinic group,waiting_time_minutes,Waiting time group,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,...,1-2 Weeks,2,0,Yes,WhatsApp,19.3,Moderate Distance,29.0,Moderate,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,...,Short Term,6,0,Yes,SMS,14.3,Moderate Distance,42.0,Long,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,...,1-2 Months,5,1,Yes,SMS,11.4,Moderate Distance,11.0,Short,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,...,1-2 Months,3,1,Yes,SMS,7.4,Close Distance,35.0,Long,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,...,1-2 Months,3,1,Yes,Email,5.6,Close Distance,27.0,Moderate,No-Show


In [12]:
print("Distance groups:")
print(df['Distance to clinic group'].value_counts(dropna=False))
print()
print("Booking lead categories:")
print(df['Booking lead days category'].value_counts(dropna=False))
print()
print("Appointment types:")
print(df['appointment_type'].value_counts(dropna=False))
print()
print("Age groups:")
print(df['age_group'].value_counts(dropna=False))

Distance groups:
Distance to clinic group
Close Distance       2848
Moderate Distance    1669
Far Distance          321
Not specified          90
Very Far Distance      72
Name: count, dtype: int64

Booking lead categories:
Booking lead days category
1-2 Months    2425
2-4 Weeks     1333
Short Term     640
1-2 Weeks      602
Name: count, dtype: int64

Appointment types:
appointment_type
General Consultation       2086
Follow-up                  1421
Specialist Consultation     900
Diagnostic Test             593
Name: count, dtype: int64

Age groups:
age_group
65+      1241
35-44     819
55-64     800
45-54     793
25-34     783
18-24     564
Name: count, dtype: int64


In [15]:
# T7 — Verify Very Far Distance + 2-4 Weeks no-show rate

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

combo = df[
    (df['Distance to clinic group'] == 'Very Far Distance') &
    (df['Booking lead days category'] == '2-4 Weeks')
]

combo_total = len(combo)
combo_no_show = (combo['appointment_outcome'] == 'No-Show').sum()
combo_rate = combo_no_show / combo_total * 100 if combo_total > 0 else 0

print(f"Very Far Distance + 2-4 Weeks:")
print(f"  Total appointments: {combo_total}")
print(f"  No-shows: {combo_no_show}")
print(f"  No-show rate: {combo_rate:.1f}%")
print(f"  Expected: 83.3%")
print(f"  Result: {'✅ PASS' if abs(combo_rate - 83.3) < 1.0 else '❌ FAIL'}")

print()
print("--- Top 6 highest-risk combinations (for cross-check) ---")

risk_combos = (
    df.groupby(['Distance to clinic group', 'Booking lead days category'])
      .apply(lambda g: pd.Series({
          'N': len(g),
          'No_Show_Rate_%': round((g['appointment_outcome'] == 'No-Show').sum() / len(g) * 100, 1)
      }))
      .reset_index()
      .query('N >= 10')  # ignore tiny groups
      .sort_values('No_Show_Rate_%', ascending=False)
      .head(6)
)

print(risk_combos.to_string(index=False))

Very Far Distance + 2-4 Weeks:
  Total appointments: 18
  No-shows: 15
  No-show rate: 83.3%
  Expected: 83.3%
  Result: ✅ PASS

--- Top 6 highest-risk combinations (for cross-check) ---
Distance to clinic group Booking lead days category     N  No_Show_Rate_%
       Very Far Distance                  2-4 Weeks  18.0            83.3
       Very Far Distance                 Short Term  10.0            80.0
            Far Distance                 1-2 Months 156.0            71.8
           Not specified                 1-2 Months  42.0            64.3
       Moderate Distance                 1-2 Months 806.0            60.3
       Very Far Distance                 1-2 Months  35.0            60.0


In [16]:
# T10 — Does SMS remain top channel across all distance bands?

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

print("=== T10: Channel Ranking by Distance Band ===\n")

distance_bands = ['Close Distance', 'Moderate Distance', 'Far Distance', 'Very Far Distance']

for band in distance_bands:
    subset = df[(df['Distance to clinic group'] == band) & (df['reminder_channel'].notna())]
    
    if len(subset) < 20:
        print(f"--- {band} (n={len(subset)}) — SKIPPED (too small) ---\n")
        continue
    
    ranking = (
        subset.groupby('reminder_channel')
              .apply(lambda g: round((g['appointment_outcome'] == 'Attended').sum() / len(g) * 100, 1))
              .sort_values(ascending=False)
    )
    
    print(f"--- {band} (n={len(subset)}) ---")
    print(ranking)
    print()

=== T10: Channel Ranking by Distance Band ===

--- Close Distance (n=2069) ---
reminder_channel
SMS         52.5
Email       50.0
WhatsApp    46.4
dtype: float64

--- Moderate Distance (n=1201) ---
reminder_channel
Email       47.1
SMS         46.4
WhatsApp    43.8
dtype: float64

--- Far Distance (n=241) ---
reminder_channel
SMS         46.0
WhatsApp    39.2
Email       25.0
dtype: float64

--- Very Far Distance (n=56) ---
reminder_channel
SMS         28.1
Email       25.0
WhatsApp    18.8
dtype: float64



In [17]:
# T11 — Does the Very Far + 2-4 Weeks risk hold across appointment types?

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

print("=== T11: Very Far Distance + 2-4 Weeks, by Appointment Type ===\n")

combo = df[
    (df['Distance to clinic group'] == 'Very Far Distance') &
    (df['Booking lead days category'] == '2-4 Weeks')
]

print(f"Total appointments in this combo: {len(combo)}\n")

if len(combo) == 0:
    print("⚠️ No appointments match this combination — cannot test T11.")
else:
    type_breakdown = (
        combo.groupby('appointment_type')
             .apply(lambda g: pd.Series({
                 'N': len(g),
                 'No_Show_Rate_%': round((g['appointment_outcome'] == 'No-Show').sum() / len(g) * 100, 1)
             }))
             .reset_index()
             .sort_values('No_Show_Rate_%', ascending=False)
    )
    print(type_breakdown.to_string(index=False))
    print()
    print("Interpretation: if the no-show rate stays high across most types, the risk combination is robust.")

=== T11: Very Far Distance + 2-4 Weeks, by Appointment Type ===

Total appointments in this combo: 18

       appointment_type   N  No_Show_Rate_%
        Diagnostic Test 3.0           100.0
              Follow-up 3.0           100.0
   General Consultation 8.0            75.0
Specialist Consultation 4.0            75.0

Interpretation: if the no-show rate stays high across most types, the risk combination is robust.


In [18]:
# T8 helper — KPI values by distance band (compare to dashboard filter)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

print("=== T8 Reference: Attendance & No-Show by Distance Band ===\n")

distance_kpi = (
    df.groupby('Distance to clinic group')
      .apply(lambda g: pd.Series({
          'N': len(g),
          'Attended': (g['appointment_outcome'] == 'Attended').sum(),
          'Attendance_Rate_%': round((g['appointment_outcome'] == 'Attended').sum() / len(g) * 100, 1),
          'No_Show_Rate_%': round((g['appointment_outcome'] == 'No-Show').sum() / len(g) * 100, 1)
      }))
      .reset_index()
)
print(distance_kpi.to_string(index=False))

print()
print("=== T9 Reference: Attendance & No-Show by Age Group ===\n")

age_kpi = (
    df.groupby('age_group')
      .apply(lambda g: pd.Series({
          'N': len(g),
          'Attended': (g['appointment_outcome'] == 'Attended').sum(),
          'Attendance_Rate_%': round((g['appointment_outcome'] == 'Attended').sum() / len(g) * 100, 1),
          'No_Show_Rate_%': round((g['appointment_outcome'] == 'No-Show').sum() / len(g) * 100, 1)
      }))
      .reset_index()
)
print(age_kpi.to_string(index=False))

=== T8 Reference: Attendance & No-Show by Distance Band ===

Distance to clinic group      N  Attended  Attendance_Rate_%  No_Show_Rate_%
          Close Distance 2848.0    1374.0               48.2            46.5
            Far Distance  321.0     127.0               39.6            55.5
       Moderate Distance 1669.0     753.0               45.1            49.4
           Not specified   90.0      39.0               43.3            52.2
       Very Far Distance   72.0      21.0               29.2            68.1

=== T9 Reference: Attendance & No-Show by Age Group ===

age_group      N  Attended  Attendance_Rate_%  No_Show_Rate_%
    18-24  564.0     257.0               45.6            50.2
    25-34  783.0     353.0               45.1            50.7
    35-44  819.0     374.0               45.7            48.4
    45-54  793.0     365.0               46.0            48.0
    55-64  800.0     360.0               45.0            50.7
      65+ 1241.0     605.0               48.8  